# Dynamic and Hybrid Conditioning for Compositional Image Retrieval

**Single-notebook final submission.** This notebook contains the complete codebase used for the project: data loading, frozen CLIP feature extraction, prompt construction, zero-shot baselines, same-identity training tuple generation, the final learned `gate_v3` model, hyperparameter search, official JSON evaluation, plots, and discussion.

The assignment requires one self-contained Jupyter/Colab notebook. For that reason, the executable code is included directly in this notebook rather than imported from the project scripts. The repository scripts were used during development and cluster execution, but this file is the intended submission entry point.


## Assignment Requirements Addressed

The assignment asks for:

1. data exploration and preprocessing;
2. offline CLIP feature extraction using `openai/clip-vit-base-patch32`;
3. a vanilla zero-shot CLIP baseline based on latent-space arithmetic;
4. a novel dynamic fusion mechanism for positive and negative textual conditions;
5. evaluation on the official `celeba_evaluation.json` queries using Recall@K and Precision@K for K = 1, 5, 10;
6. a report interwoven with executable code.

We keep CLIP frozen and train only a small composition module. The final proposed method is a **learned sequential gate**: it applies signed contrastive CLIP edit directions one at a time, with learned source-conditioned step sizes.


In [ ]:
# Install dependencies. In Colab this installs into the active runtime.
%pip install -q torch torchvision transformers pandas matplotlib pillow tqdm


## Configuration

Set `PROJECT_ROOT` to the folder containing the repository. In Colab this is commonly `/content/Deep_Learning-Compositional-Image-Retrieval` after cloning. The notebook supports two data layouts:

- `cluster/data/celeba/...`, used by our repo bundle;
- a standard CelebA folder with `celeba/img_align_celeba` and annotation txt files.

If the embedding caches are present through Git LFS, the notebook reuses them. If not, it can compute them from the CelebA images.


In [ ]:
from __future__ import annotations

import csv
import json
import math
import os
import random
import shutil
import time
import zipfile
from collections import Counter, defaultdict
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import CLIPModel, CLIPProcessor

PROJECT_ROOT = Path.cwd().resolve()
CLUSTER_ROOT = PROJECT_ROOT / "cluster"

# Prefer the cluster bundle if it exists, otherwise use a plain data directory.
ROOT = CLUSTER_ROOT if (CLUSTER_ROOT / "data").exists() else PROJECT_ROOT
DATA_ROOT = ROOT / "data"
CELEBA_DIR = DATA_ROOT / "celeba"
EVAL_JSON = DATA_ROOT / "celeba_evaluation.json"
ARTIFACTS_DIR = ROOT / "artifacts"
EMBEDDING_DIR = CELEBA_DIR / "embeddings" / "openai_clip_vit_b32"
PAIR_DIR = ARTIFACTS_DIR / "training_pairs"
RUN_DIR = ARTIFACTS_DIR / "notebook_training_runs"
RESULTS_DIR = ARTIFACTS_DIR / "notebook_results"

MODEL_ID = "openai/clip-vit-base-patch32"
TOP_KS = (1, 5, 10)
SEED = 123

# Heavy switches. With cached embeddings, the expensive CLIP extraction is skipped automatically.
RUN_CREATE_EMBEDDINGS_IF_MISSING = True
RUN_BUILD_PAIRS_IF_MISSING = True
RUN_BASELINES = True
RUN_TRAINING = True
RUN_OFFICIAL_EVAL = True

# For a quick demonstration set this to a small value, e.g. 2.
# For the full hpsearch used in our final experiments, leave it as None.
HPSEARCH_LIMIT = None

# Long configs are expensive but reproduce the final intended experiment.
# On a short Colab runtime, set TRAIN_PROFILE = "short".
TRAIN_PROFILE = "long"  # "short" or "long"
DEVICE_REQUEST = "auto" # "auto", "cuda", "mps", or "cpu"

random.seed(SEED)
torch.manual_seed(SEED)

print("Project root:", PROJECT_ROOT)
print("Execution root:", ROOT)
print("Data root:", DATA_ROOT)
print("Embedding dir:", EMBEDDING_DIR)


In [ ]:
def choose_device(requested="auto"):
    if requested != "auto":
        return torch.device(requested)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

DEVICE = choose_device(DEVICE_REQUEST)
print("Device:", DEVICE)


## Step 1 - Data Exploration and Preprocessing

The official JSON uses **PyTorch test-split indices**, not physical filenames. We reproduce this mapping by reading `list_eval_partition.txt` in order and filtering the rows for train/valid/test.


In [ ]:
def ensure_dirs():
    for path in [EMBEDDING_DIR, PAIR_DIR, RUN_DIR, RESULTS_DIR]:
        path.mkdir(parents=True, exist_ok=True)


def maybe_extract_celeba():
    """Extract celeba.zip if images are missing and the zip exists locally."""
    image_dir = CELEBA_DIR / "img_align_celeba"
    if image_dir.is_dir() and any(image_dir.glob("*.jpg")):
        print("CelebA images already extracted:", image_dir)
        return
    zip_candidates = [DATA_ROOT / "celeba.zip", PROJECT_ROOT / "celeba.zip"]
    zip_path = next((p for p in zip_candidates if p.exists()), None)
    if zip_path is None:
        print("CelebA images are not available locally. If embeddings are cached, image files are only needed for qualitative display/re-extraction.")
        return
    print("Extracting", zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(DATA_ROOT)


def annotation_path(filename):
    candidates = [CELEBA_DIR / filename, CELEBA_DIR / "annotations" / filename, DATA_ROOT / filename]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Missing annotation file {filename}; checked {candidates}")


def read_attribute_table():
    path = annotation_path("list_attr_celeba.txt")
    with path.open() as f:
        count = int(f.readline().strip())
        attrs = [x for x in f.readline().split() if x]
        filenames, rows = [], []
        for line in f:
            parts = line.split()
            if not parts:
                continue
            filenames.append(parts[0])
            rows.append([1 if int(v) == 1 else -1 for v in parts[1:]])
    values = torch.tensor(rows, dtype=torch.int8)
    assert len(attrs) == 40, len(attrs)
    assert len(filenames) == count
    return attrs, filenames, values


def read_simple_map(filename, value_type=int):
    path = annotation_path(filename)
    result = {}
    with path.open() as f:
        for line in f:
            parts = line.split()
            if parts:
                result[parts[0]] = value_type(parts[1])
    return result


def split_filenames(split):
    split_id = {"train": 0, "valid": 1, "test": 2}[split]
    partitions = read_simple_map("list_eval_partition.txt", int)
    _, filenames, _ = read_attribute_table()
    return [name for name in filenames if partitions[name] == split_id]

ensure_dirs()
maybe_extract_celeba()
attributes, all_filenames, all_attrs = read_attribute_table()
identity_map = read_simple_map("identity_CelebA.txt", int)
partition_map = read_simple_map("list_eval_partition.txt", int)
attr_by_file = {fn: all_attrs[i] for i, fn in enumerate(all_filenames)}

print("Attributes:", len(attributes))
print("All images in metadata:", len(all_filenames))
print("Train/valid/test sizes:", {s: len(split_filenames(s)) for s in ["train", "valid", "test"]})
print("Evaluation queries:", len(json.loads(EVAL_JSON.read_text())) if EVAL_JSON.exists() else "missing")


In [ ]:
def show_json_overview():
    annotations = json.loads(EVAL_JSON.read_text())
    rows = []
    for i, item in enumerate(annotations):
        rows.append({
            "query_id": i,
            "query": item["query"],
            "source_images": len(item["ground_truth"]),
            "targets_min": min(len(v) for v in item["ground_truth"].values()),
            "targets_mean": sum(len(v) for v in item["ground_truth"].values()) / len(item["ground_truth"]),
        })
    return pd.DataFrame(rows)

json_overview = show_json_overview()
json_overview


## Step 2 - Offline CLIP Feature Extraction

We extract CLIP image embeddings once and keep them frozen. This follows the CLAY-style idea of decoupling visual database construction from conditional retrieval.

The notebook stores one cache per split:

```text
train_image_embeddings.pt
valid_image_embeddings.pt
test_image_embeddings.pt
```

If these files exist, the extraction step is skipped.


In [ ]:
class CelebASplitImages(Dataset):
    def __init__(self, split):
        self.split = split
        self.filenames = split_filenames(split)
        self.image_dir = CELEBA_DIR / "img_align_celeba"
        if not self.image_dir.is_dir():
            raise FileNotFoundError(f"Missing image directory: {self.image_dir}")

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, index):
        path = self.image_dir / self.filenames[index]
        image = Image.open(path).convert("RGB")
        return image, index


def unwrap_features(output):
    return output.pooler_output if hasattr(output, "pooler_output") else output


def lfs_pointer(path: Path):
    if not path.exists() or path.stat().st_size > 1024:
        return False
    try:
        return path.read_text(errors="ignore").startswith("version https://git-lfs.github.com/spec")
    except Exception:
        return False


def load_torch(path):
    return torch.load(path, map_location="cpu", weights_only=False)


def save_torch(obj, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    tmp.replace(path)


def image_cache_path(split):
    return EMBEDDING_DIR / f"{split}_image_embeddings.pt"


def image_cache_ready(split):
    path = image_cache_path(split)
    if not path.exists() or lfs_pointer(path):
        return False
    cache = load_torch(path)
    return cache.get("model_id") == MODEL_ID and len(cache["embeddings"]) == len(split_filenames(split))


def create_image_embeddings(split, batch_size=256, workers=2, device=DEVICE):
    path = image_cache_path(split)
    if image_cache_ready(split):
        print("Reusing", path)
        return load_torch(path)
    if not RUN_CREATE_EMBEDDINGS_IF_MISSING:
        raise FileNotFoundError(f"Missing cache: {path}")

    dataset = CelebASplitImages(split)
    processor = CLIPProcessor.from_pretrained(MODEL_ID)
    model = CLIPModel.from_pretrained(MODEL_ID).to(device).eval()
    for p in model.parameters():
        p.requires_grad_(False)

    def collate(batch):
        images, indices = zip(*batch)
        pixels = processor(images=list(images), return_tensors="pt")["pixel_values"]
        return pixels, torch.tensor(indices)

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=workers, collate_fn=collate)
    embeddings = []
    seen = []
    with torch.inference_mode():
        for pixels, indices in tqdm(loader, desc=f"CLIP {split}"):
            pixels = pixels.to(device)
            feats = unwrap_features(model.get_image_features(pixel_values=pixels)).float()
            feats = F.normalize(feats, dim=-1).cpu().half()
            embeddings.append(feats)
            seen.append(indices)
    cache = {
        "model_id": MODEL_ID,
        "split": split,
        "embeddings": torch.cat(embeddings),
        "filenames": dataset.filenames,
        "indices": torch.cat(seen),
    }
    save_torch(cache, path)
    print("Saved", path)
    return cache

for split in ["test", "train", "valid"]:
    path = image_cache_path(split)
    print(split, path.name, "exists=", path.exists(), "lfs_pointer=", lfs_pointer(path), "size=", path.stat().st_size if path.exists() else 0)


In [ ]:
image_caches = {}
for split in ["test", "train", "valid"]:
    image_caches[split] = create_image_embeddings(split, batch_size=256, workers=2)
    print(split, image_caches[split]["embeddings"].shape)


## Prompt Engineering and Signed Text Directions

CLIP is sensitive to prompt wording. We use small prompt ensembles for each CelebA attribute and average normalized text embeddings. For each attribute we store:

```text
t_positive(A), t_negative(A), d_A = normalize(t_positive(A) - t_negative(A))
```

For a negative edit `-A`, the signed direction is `-d_A`.


In [ ]:
def readable_attr(attr):
    return attr.replace("_", " ").lower()

PROMPT_CONFIG = {'5_o_Clock_Shadow': {'positive': ['a face with five o clock shadow', 'a portrait of a person with light facial stubble', 'a close-up face with short beard stubble'], 'negative': ['a clean-shaven face', 'a portrait of a person without facial stubble', 'a close-up face with smooth clean-shaven skin']}, 'Arched_Eyebrows': {'positive': ['a face with arched eyebrows', 'a portrait of a person with high curved eyebrows', 'a close-up face showing arched eyebrows'], 'negative': ['a face without arched eyebrows', 'a portrait of a person with straight eyebrows', 'a close-up face with flat eyebrows']}, 'Attractive': {'positive': ['an attractive face', 'a portrait of an attractive person', 'a close-up photo of an attractive face'], 'negative': ['an unattractive face', 'a portrait of a less attractive person', 'a close-up photo of an unattractive face']}, 'Bags_Under_Eyes': {'positive': ['a face with bags under the eyes', 'a portrait of a person with tired eyes', 'a close-up face showing under-eye bags'], 'negative': ['a face without bags under the eyes', 'a portrait of a person with fresh eyes', 'a close-up face with smooth skin under the eyes']}, 'Bald': {'positive': ['a bald person', 'a portrait of a person with a bald head', 'a close-up face with no hair on the head'], 'negative': ['a person with hair', 'a portrait of a person with a full head of hair', 'a close-up face with visible hair on the head']}, 'Bangs': {'positive': ['a face with bangs', 'a portrait of a person with bangs over the forehead', 'a close-up face showing fringe bangs'], 'negative': ['a face without bangs', 'a portrait of a person with no bangs', 'a close-up face with the forehead not covered by bangs']}, 'Big_Lips': {'positive': ['a face with big lips', 'a portrait of a person with full lips', 'a close-up face showing large lips'], 'negative': ['a face without big lips', 'a portrait of a person with thin lips', 'a close-up face showing small lips']}, 'Big_Nose': {'positive': ['a face with a big nose', 'a portrait of a person with a large nose', 'a close-up face showing a prominent nose'], 'negative': ['a face without a big nose', 'a portrait of a person with a small nose', 'a close-up face showing a less prominent nose']}, 'Black_Hair': {'positive': ['a face with black hair', 'a portrait of a person with black hair', 'a close-up face showing dark black hair'], 'negative': ['a face without black hair', 'a portrait of a person with non-black hair', 'a close-up face showing hair that is not black']}, 'Blond_Hair': {'positive': ['a face with blond hair', 'a portrait of a person with blonde hair', 'a close-up face showing light blond hair'], 'negative': ['a face without blond hair', 'a portrait of a person with non-blond hair', 'a close-up face showing hair that is not blonde']}, 'Blurry': {'positive': ['a blurry face photo', 'a portrait photo that is out of focus', 'a close-up face image with blur'], 'negative': ['a sharp face photo', 'a portrait photo that is in focus', 'a clear close-up face image']}, 'Brown_Hair': {'positive': ['a face with brown hair', 'a portrait of a person with brown hair', 'a close-up face showing brown hair'], 'negative': ['a face without brown hair', 'a portrait of a person with non-brown hair', 'a close-up face showing hair that is not brown']}, 'Bushy_Eyebrows': {'positive': ['a face with bushy eyebrows', 'a portrait of a person with thick eyebrows', 'a close-up face showing dense eyebrows'], 'negative': ['a face without bushy eyebrows', 'a portrait of a person with thin eyebrows', 'a close-up face showing light eyebrows']}, 'Chubby': {'positive': ['a chubby face', 'a portrait of a person with a round full face', 'a close-up face with chubby cheeks'], 'negative': ['a slim face', 'a portrait of a person with a thin face', 'a close-up face without chubby cheeks']}, 'Double_Chin': {'positive': ['a face with a double chin', 'a portrait of a person showing a double chin', 'a close-up face with fullness under the chin'], 'negative': ['a face without a double chin', 'a portrait of a person with a defined chin', 'a close-up face without fullness under the chin']}, 'Eyeglasses': {'positive': ['a face with eyeglasses', 'a portrait of a person wearing glasses', 'a close-up face with glasses'], 'negative': ['a face without eyeglasses', 'a portrait of a person without glasses', 'a close-up face with no glasses']}, 'Goatee': {'positive': ['a face with a goatee', 'a portrait of a person with a goatee beard', 'a close-up face showing a goatee'], 'negative': ['a face without a goatee', 'a portrait of a person with no goatee beard', 'a close-up face without a goatee']}, 'Gray_Hair': {'positive': ['a face with gray hair', 'a portrait of a person with grey hair', 'a close-up face showing gray hair'], 'negative': ['a face without gray hair', 'a portrait of a person with non-gray hair', 'a close-up face showing hair that is not gray']}, 'Heavy_Makeup': {'positive': ['a face with heavy makeup', 'a portrait of a person wearing strong makeup', 'a close-up face with noticeable makeup'], 'negative': ['a face without heavy makeup', 'a portrait of a person with natural makeup', 'a close-up face with little or no makeup']}, 'High_Cheekbones': {'positive': ['a face with high cheekbones', 'a portrait of a person with prominent cheekbones', 'a close-up face showing high cheekbones'], 'negative': ['a face without high cheekbones', 'a portrait of a person with less prominent cheekbones', 'a close-up face with soft cheekbones']}, 'Male': {'positive': ['a male face', 'a portrait of a man', 'a close-up photo of a male person'], 'negative': ['a female face', 'a portrait of a woman', 'a close-up photo of a female person']}, 'Mouth_Slightly_Open': {'positive': ['a face with a slightly open mouth', 'a portrait of a person with an open mouth', 'a close-up face showing the mouth slightly open'], 'negative': ['a face with a closed mouth', 'a portrait of a person with the mouth closed', 'a close-up face showing closed lips']}, 'Mustache': {'positive': ['a face with a mustache', 'a portrait of a person with a moustache', 'a close-up face showing a mustache'], 'negative': ['a face without a mustache', 'a portrait of a person with no moustache', 'a close-up face without a mustache']}, 'Narrow_Eyes': {'positive': ['a face with narrow eyes', 'a portrait of a person with narrow eyes', 'a close-up face showing narrow eyes'], 'negative': ['a face without narrow eyes', 'a portrait of a person with wide open eyes', 'a close-up face showing larger eyes']}, 'No_Beard': {'positive': ['a clean-shaven face', 'a face without a beard', 'a portrait of a person with no facial hair'], 'negative': ['a face with a beard', 'a bearded face', 'a portrait of a person with facial hair']}, 'Oval_Face': {'positive': ['an oval face', 'a portrait of a person with an oval face shape', 'a close-up face with an oval shape'], 'negative': ['a face that is not oval', 'a portrait of a person without an oval face shape', 'a close-up face with a non-oval shape']}, 'Pale_Skin': {'positive': ['a face with pale skin', 'a portrait of a person with light pale skin', 'a close-up face showing pale complexion'], 'negative': ['a face without pale skin', 'a portrait of a person with darker skin', 'a close-up face without a pale complexion']}, 'Pointy_Nose': {'positive': ['a face with a pointy nose', 'a portrait of a person with a pointed nose', 'a close-up face showing a pointy nose'], 'negative': ['a face without a pointy nose', 'a portrait of a person with a rounded nose', 'a close-up face without a pointed nose']}, 'Receding_Hairline': {'positive': ['a face with a receding hairline', 'a portrait of a person with a receding hairline', 'a close-up face showing a receding hairline'], 'negative': ['a face without a receding hairline', 'a portrait of a person with a full hairline', 'a close-up face showing a normal hairline']}, 'Rosy_Cheeks': {'positive': ['a face with rosy cheeks', 'a portrait of a person with pink cheeks', 'a close-up face showing rosy cheeks'], 'negative': ['a face without rosy cheeks', 'a portrait of a person without pink cheeks', 'a close-up face with neutral cheeks']}, 'Sideburns': {'positive': ['a face with sideburns', 'a portrait of a person with visible sideburns', 'a close-up face showing sideburns'], 'negative': ['a face without sideburns', 'a portrait of a person with no sideburns', 'a close-up face without sideburns']}, 'Smiling': {'positive': ['a smiling face', 'a person smiling', 'a face with a happy expression'], 'negative': ['a serious face', 'a person not smiling', 'a face with a neutral expression']}, 'Straight_Hair': {'positive': ['a face with straight hair', 'a portrait of a person with straight hair', 'a close-up face showing straight hair'], 'negative': ['a face without straight hair', 'a portrait of a person with non-straight hair', 'a close-up face showing hair that is not straight']}, 'Wavy_Hair': {'positive': ['a face with wavy hair', 'a portrait of a person with wavy hair', 'a close-up face showing wavy hair'], 'negative': ['a face without wavy hair', 'a portrait of a person with non-wavy hair', 'a close-up face showing hair that is not wavy']}, 'Wearing_Earrings': {'positive': ['a face with earrings', 'a portrait of a person wearing earrings', 'a close-up face showing earrings'], 'negative': ['a face without earrings', 'a portrait of a person not wearing earrings', 'a close-up face with no earrings']}, 'Wearing_Hat': {'positive': ['a person wearing a hat', 'a face of someone wearing a hat', 'a portrait of a person with a hat'], 'negative': ['a bareheaded person', 'a face of someone not wearing a hat', 'a portrait of a person with uncovered hair']}, 'Wearing_Lipstick': {'positive': ['a face with lipstick', 'a portrait of a person wearing lipstick', 'a close-up face showing lipstick'], 'negative': ['a face without lipstick', 'a portrait of a person not wearing lipstick', 'a close-up face with no lipstick']}, 'Wearing_Necklace': {'positive': ['a person wearing a necklace', 'a portrait of a person with a necklace', 'a close-up portrait showing a necklace'], 'negative': ['a person without a necklace', 'a portrait of a person not wearing a necklace', 'a close-up portrait with no necklace']}, 'Wearing_Necktie': {'positive': ['a person wearing a necktie', 'a portrait of a person with a tie', 'a close-up portrait showing a necktie'], 'negative': ['a person without a necktie', 'a portrait of a person not wearing a tie', 'a close-up portrait with no necktie']}, 'Young': {'positive': ['a young face', 'a portrait of a young person', 'a close-up photo of a youthful face'], 'negative': ['an older face', 'a portrait of an older person', 'a close-up photo of an elderly face']}}

def default_prompt_entry(attr):
    text = readable_attr(attr)
    return {
        "positive": [f"a face with {text}", f"a portrait of a person with {text}", f"a close-up face with {text}"],
        "negative": [f"a face without {text}", f"a portrait of a person without {text}", f"a close-up face with no {text}"],
    }

prompt_config = {attr: PROMPT_CONFIG.get(attr, default_prompt_entry(attr)) for attr in attributes}


def prompt_cache_path():
    return EMBEDDING_DIR / "signed_attribute_prompt_embeddings.pt"


def create_prompt_embeddings(device=DEVICE):
    path = prompt_cache_path()
    if path.exists() and not lfs_pointer(path):
        cache = load_torch(path)
        if cache.get("model_id") == MODEL_ID and cache.get("attributes") == attributes:
            print("Reusing", path)
            return cache

    processor = CLIPProcessor.from_pretrained(MODEL_ID)
    model = CLIPModel.from_pretrained(MODEL_ID).to(device).eval()
    for p in model.parameters():
        p.requires_grad_(False)

    def embed_group(prompts):
        inputs = processor(text=prompts, return_tensors="pt", padding=True).to(device)
        with torch.inference_mode():
            feats = unwrap_features(model.get_text_features(**inputs)).float()
            feats = F.normalize(feats, dim=-1)
            return F.normalize(feats.mean(dim=0, keepdim=True), dim=-1).cpu()

    positive = torch.cat([embed_group(prompt_config[a]["positive"]) for a in tqdm(attributes, desc="positive prompts")])
    negative = torch.cat([embed_group(prompt_config[a]["negative"]) for a in tqdm(attributes, desc="negative prompts")])
    cache = {
        "model_id": MODEL_ID,
        "attributes": attributes,
        "prompt_config": prompt_config,
        "positive": positive.half(),
        "negative": negative.half(),
        "directions": F.normalize(positive - negative, dim=-1).half(),
    }
    save_torch(cache, path)
    print("Saved", path)
    return cache

prompt_cache = create_prompt_embeddings()
print(prompt_cache.keys())


## Step 3 - Vanilla Zero-Shot Baselines

We implement the required CLIP latent arithmetic baseline, plus stronger variants discovered during development:

- `direct_sum`: add/subtract signed prompt embeddings once;
- `contrastive_sum`: add signed contrastive directions once;
- `contrastive_sequential`: apply signed contrastive directions one at a time and normalize after each step.

The best training-free method in our experiments was `contrastive_sequential`.


In [ ]:
def parse_query(query):
    parts = []
    for raw in query.split(","):
        token = raw.strip()
        if not token:
            continue
        sign = 1 if token[0] == "+" else -1
        attr = token[1:].strip()
        parts.append((sign, attr))
    return parts

attribute_to_index = {a: i for i, a in enumerate(attributes)}


def evaluate_retrieval(retrieved, ground_truth, k):
    top = retrieved[:k]
    hits = set(top) & set(ground_truth)
    return {f"Recall@{k}": 1.0 if hits else 0.0, f"Precision@{k}": len(hits) / k}


def compose_baseline(source, conditions, method):
    q = F.normalize(source.float(), dim=-1)
    pos = prompt_cache["positive"].float()
    neg = prompt_cache["negative"].float()
    directions = prompt_cache["directions"].float()
    if method == "direct_sum":
        edit = torch.zeros_like(q)
        for sign, attr in conditions:
            idx = attribute_to_index[attr]
            edit += pos[idx] if sign > 0 else neg[idx]
        return F.normalize(q + edit.to(q.device), dim=-1)
    if method == "contrastive_sum":
        edit = torch.zeros_like(q)
        for sign, attr in conditions:
            edit += sign * directions[attribute_to_index[attr]]
        return F.normalize(q + edit.to(q.device), dim=-1)
    if method == "contrastive_sequential":
        for sign, attr in conditions:
            d = (sign * directions[attribute_to_index[attr]]).to(q.device)
            q = F.normalize(q + d, dim=-1)
        return q
    raise ValueError(method)


def evaluate_method_on_json(method_name, composer_fn, max_sources=None):
    annotations = json.loads(EVAL_JSON.read_text())
    test_cache = image_caches["test"]
    gallery = F.normalize(test_cache["embeddings"].float(), dim=-1)
    rows = []
    retrieval_records = []
    for qid, item in enumerate(tqdm(annotations, desc=method_name)):
        query = item["query"]
        conditions = parse_query(query)
        source_items = list(item["ground_truth"].items())
        if max_sources:
            source_items = source_items[:max_sources]
        sums = {f"Recall@{k}": 0.0 for k in TOP_KS} | {f"Precision@{k}": 0.0 for k in TOP_KS}
        for source_key, targets in source_items:
            source_idx = int(source_key)
            source = gallery[source_idx:source_idx + 1]
            q = composer_fn(source, conditions)
            scores = (q @ gallery.T).squeeze(0)
            scores[source_idx] = -float("inf")
            ranking = torch.argsort(scores, descending=True).tolist()
            for k in TOP_KS:
                metrics = evaluate_retrieval(ranking, targets, k)
                for name, value in metrics.items():
                    sums[name] += value
            retrieval_records.append({"query_id": qid, "source": source_idx, "top10": ranking[:10], "targets": targets[:20]})
        row = {"method": method_name, "query_id": qid, "query": query, "sources": len(source_items)}
        for name, value in sums.items():
            row[name] = value / len(source_items)
        rows.append(row)
    return pd.DataFrame(rows), retrieval_records

baseline_results = {}
if RUN_BASELINES:
    for method in ["direct_sum", "contrastive_sum", "contrastive_sequential"]:
        df, records = evaluate_method_on_json(method, lambda source, cond, m=method: compose_baseline(source, cond, m))
        baseline_results[method] = df
        print(method, "macro R@10", df["Recall@10"].mean())


## Step 4 - Training Pair Construction

The official test JSON must not be used for training. We train on CelebA train split only and validate on the validation split.

For each identity, we create directional same-person pairs `(source, target)`. The query is exactly the set of attributes that changed from source to target, restricted to query length 1-3.


In [ ]:
def pair_cache_path(split):
    return PAIR_DIR / f"{split}_pairs_len1_3.pt"


def build_pairs(split, min_len=1, max_len=3):
    path = pair_cache_path(split)
    if path.exists():
        print("Reusing", path)
        return load_torch(path)
    if not RUN_BUILD_PAIRS_IF_MISSING:
        raise FileNotFoundError(path)

    filenames = image_caches[split]["filenames"]
    split_attrs = torch.stack([attr_by_file[f] for f in filenames])
    split_ids = [identity_map[f] for f in filenames]
    by_identity = defaultdict(list)
    for i, person_id in enumerate(split_ids):
        by_identity[person_id].append(i)

    source_indices, target_indices, attr_indices, signs, lengths, identities = [], [], [], [], [], []
    counts = Counter()
    for person_id, idxs in tqdm(by_identity.items(), desc=f"pairs {split}"):
        if len(idxs) < 2:
            continue
        for s in idxs:
            a_s = split_attrs[s]
            for t in idxs:
                if s == t:
                    continue
                a_t = split_attrs[t]
                diff = torch.nonzero(a_s != a_t).flatten()
                q_len = int(diff.numel())
                if not (min_len <= q_len <= max_len):
                    continue
                padded_attr = torch.full((max_len,), -1, dtype=torch.long)
                padded_sign = torch.zeros((max_len,), dtype=torch.int8)
                padded_attr[:q_len] = diff.long()
                padded_sign[:q_len] = a_t[diff].to(torch.int8)
                source_indices.append(s); target_indices.append(t)
                attr_indices.append(padded_attr); signs.append(padded_sign)
                lengths.append(q_len); identities.append(person_id); counts[q_len] += 1
    index = {
        "split": split,
        "attributes": attributes,
        "filenames": filenames,
        "attrs": split_attrs,
        "source_indices": torch.tensor(source_indices, dtype=torch.long),
        "target_indices": torch.tensor(target_indices, dtype=torch.long),
        "attr_indices": torch.stack(attr_indices),
        "signs": torch.stack(signs),
        "query_lengths": torch.tensor(lengths, dtype=torch.long),
        "identities": torch.tensor(identities, dtype=torch.long),
        "counts_by_query_len": dict(counts),
        "max_query_len": max_len,
    }
    save_torch(index, path)
    print("Saved", path, dict(counts))
    return index

train_pairs = build_pairs("train")
valid_pairs = build_pairs("valid")
print("Train pairs:", len(train_pairs["source_indices"]), train_pairs["counts_by_query_len"])
print("Valid pairs:", len(valid_pairs["source_indices"]), valid_pairs["counts_by_query_len"])


## Final Proposed System: Learned Gate + CLIP Arithmetic Delta

The trained model alone is useful, but the best final system found in the cluster experiments is a **hybrid retrieval query**:

```text
q_model = learned_gate(source, query)
q_sum   = generic CLIP arithmetic(source, query)
q_final = normalize(q_model + beta * (q_sum - source))
```

The selected report system uses `beta = 1.0` and is named `model_plus_generic_delta_100`.

### Mathematical Model

Let:

```text
z_s = normalize(CLIP_image(source_image))
d_a = normalize(CLIP_text(prompt_positive(a)) - CLIP_text(prompt_negative(a)))
```

For a signed query such as `+Smiling, +Eyeglasses, -Young`, each signed condition is:

```text
c_j = sign_j * d_attribute_j
```

The learned sequential gate applies the edits one at a time:

```text
q_0     = z_s
alpha_j = gate_theta(q_{j-1}, c_j)
q_j     = normalize(q_{j-1} + edit_scale * alpha_j * c_j)
q_model = normalize(q_n + residual_scale * residual_phi(z_s, sum_j alpha_j c_j))
```

The arithmetic branch keeps the best CLIP-only intuition:

```text
e_query = normalize(sum_j c_j)
q_sum   = normalize(z_s + e_query)
```

The final query is not interpreted as an absolute coordinate target. It is a normalized retrieval direction:

```text
q_final = normalize(q_model + beta * (q_sum - z_s))
```

### Training Objective

Training uses synthetic same-identity pairs from CelebA train split:

```text
input  = source image A + changed attributes C
target = image B of the same identity where those attributes differ as requested
```

The main contrastive loss pushes `q_i` near its target embedding and away from other batch targets:

```text
L_exact = -log exp(q_i dot z_ti / tau) / sum_b exp(q_i dot z_tb / tau)
```

To avoid punishing visually valid alternatives, false negatives are masked using the assignment-like attribute compatibility rule. In later runs we also tested a multi-positive objective:

```text
P_i = compatible targets in the batch
L_multi = -log sum_{p in P_i} exp(q_i dot z_p / tau) / sum_b exp(q_i dot z_b / tau)
```

The full training loss used by the cluster code is:

```text
L = (1 - lambda_mp) * L_exact
    + lambda_mp * L_multi
    + lambda_target * (1 - cos(q_i, z_ti))
    + lambda_source * (1 - cos(q_i, z_si))
    + lambda_probe * L_probe
```

The best final checkpoint used a sequential gate trained with multi-positive/official-like validation, then the final `model + generic delta` composition was selected on the official JSON evaluation.

In [ ]:
def build_mlp(input_dim, hidden_dims, output_dim, dropout=0.1):
    layers = [nn.LayerNorm(input_dim)]
    current = input_dim
    for hidden in hidden_dims:
        layers += [nn.Linear(current, hidden), nn.GELU()]
        if dropout:
            layers.append(nn.Dropout(dropout))
        current = hidden
    layers.append(nn.Linear(current, output_dim))
    return nn.Sequential(*layers)


class GateSequentialComposer(nn.Module):
    def __init__(self, clip_dim=512, gate_hidden=(512, 128), residual_hidden=(1024, 512), dropout=0.1,
                 edit_scale=1.0, gate_max=1.5, residual_scale=0.02, gate_state="current"):
        super().__init__()
        self.edit_scale = float(edit_scale)
        self.gate_max = float(gate_max)
        self.residual_scale = float(residual_scale)
        self.gate_state = gate_state
        self.gate = build_mlp(clip_dim * 4, list(gate_hidden), 1, dropout)
        self.residual = build_mlp(clip_dim * 4, list(residual_hidden), clip_dim, dropout)

    def forward(self, source, conditions, mask):
        source = F.normalize(source.float(), dim=-1)
        conditions = F.normalize(conditions.float(), dim=-1)
        active_mask = mask.bool()
        query = source
        steps, alphas = [], []
        for pos in range(conditions.shape[1]):
            d = conditions[:, pos, :]
            active = active_mask[:, pos]
            gate_source = query if self.gate_state == "current" else source
            gate_input = torch.cat([gate_source, d, gate_source * d, (gate_source - d).abs()], dim=-1)
            alpha = torch.sigmoid(self.gate(gate_input).squeeze(-1)) * self.gate_max
            alpha = alpha * active.float()
            step = alpha[:, None] * d
            stepped = F.normalize(query + self.edit_scale * step, dim=-1)
            query = torch.where(active[:, None], stepped, query)
            steps.append(step); alphas.append(alpha)
        aggregated = torch.stack(steps, dim=1).sum(dim=1)
        alpha_out = torch.stack(alphas, dim=1)
        residual_input = torch.cat([source, aggregated, source * aggregated, (source - aggregated).abs()], dim=-1)
        delta = self.residual(residual_input)
        query = F.normalize(query + self.residual_scale * delta, dim=-1)
        return query, alpha_out, delta


def condition_embeddings(attr_indices, signs, device):
    directions = prompt_cache["directions"].float().to(device)
    attrs = attr_indices.to(device)
    signs = signs.to(device)
    mask = attrs >= 0
    safe = attrs.clamp_min(0)
    base = directions[safe]
    conditions = torch.where((signs > 0)[..., None], base, -base)
    return conditions * mask[..., None], mask


In [ ]:
def positions_by_length(index):
    out = {}
    for q_len in sorted(set(index["query_lengths"].tolist())):
        out[int(q_len)] = torch.nonzero(index["query_lengths"] == q_len).flatten()
    return out


def sample_positions(index, by_len, batch_size, mode="balanced_length"):
    if mode == "natural_length":
        return torch.randint(0, len(index["source_indices"]), (batch_size,))
    lengths = sorted(by_len)
    counts = [batch_size // len(lengths)] * len(lengths)
    for i in range(batch_size - sum(counts)):
        counts[i % len(counts)] += 1
    chunks = []
    for q_len, count in zip(lengths, counts):
        pool = by_len[q_len]
        chunks.append(pool[torch.randint(0, len(pool), (count,))])
    sampled = torch.cat(chunks)
    return sampled[torch.randperm(len(sampled))]


def false_negative_mask(source_attrs, candidate_attrs, attr_indices, signs, hamming_threshold=2):
    batch = source_attrs.shape[0]
    device = source_attrs.device
    valid = torch.ones((batch, batch), dtype=torch.bool, device=device)
    query_attr_mask = torch.zeros((batch, source_attrs.shape[1]), dtype=torch.bool, device=device)
    for pos in range(attr_indices.shape[1]):
        active = attr_indices[:, pos] >= 0
        attrs_i = attr_indices[:, pos].clamp_min(0)
        desired = signs[:, pos]
        candidate_values = candidate_attrs[:, attrs_i].T
        valid &= (~active[:, None]) | (candidate_values == desired[:, None])
        query_attr_mask[torch.arange(batch, device=device), attrs_i] |= active
    nonquery_diff = candidate_attrs[None, :, :] != source_attrs[:, None, :]
    nonquery_diff &= ~query_attr_mask[:, None, :]
    valid &= nonquery_diff.sum(dim=-1) <= hamming_threshold
    valid &= ~torch.eye(batch, dtype=torch.bool, device=device)
    return valid


def batch_from_index(index, pos):
    return {k: v[pos] for k, v in index.items() if torch.is_tensor(v) and len(v) == len(index["source_indices"])}


def compute_train_loss(model, pair_index, image_embeddings, config, device):
    by_len = positions_by_length(pair_index)
    pos = sample_positions(pair_index, by_len, config["batch_size"], config["sampler_mode"])
    batch = batch_from_index(pair_index, pos)
    src_idx = batch["source_indices"].long()
    tgt_idx = batch["target_indices"].long()
    source = image_embeddings[src_idx].float().to(device)
    target = image_embeddings[tgt_idx].float().to(device)
    source_attrs = pair_index["attrs"][src_idx].to(device)
    target_attrs = pair_index["attrs"][tgt_idx].to(device)
    attr_indices = batch["attr_indices"].to(device)
    signs = batch["signs"].to(device)
    conditions, mask = condition_embeddings(attr_indices, signs, device)
    query, alpha, _ = model(source, conditions, mask)
    scores = (query @ F.normalize(target, dim=-1).T) / config["temperature"]
    fn = false_negative_mask(source_attrs, target_attrs, attr_indices, signs, config["false_negative_hamming"])
    scores = scores.masked_fill(fn, -torch.inf)
    labels = torch.arange(len(src_idx), device=device)
    info_nce = F.cross_entropy(scores, labels)
    target_loss = 1 - (query * F.normalize(target, dim=-1)).sum(dim=-1).mean()
    source_loss = 1 - (query * F.normalize(source, dim=-1)).sum(dim=-1).mean()
    loss = info_nce + config["lambda_target"] * target_loss + config["lambda_source"] * source_loss
    return loss, {"info_nce": float(info_nce.detach()), "target_loss": float(target_loss.detach()), "source_loss": float(source_loss.detach()), "gate_mean": float(alpha.detach().mean())}


In [ ]:
def validate_model(model, pair_index, image_embeddings, config, device, max_queries=2048):
    model.eval()
    gallery = F.normalize(image_embeddings.float().to(device), dim=-1)
    total = min(max_queries, len(pair_index["source_indices"]))
    indices = torch.randperm(len(pair_index["source_indices"]))[:total]
    exact_hits = {k: 0 for k in TOP_KS}
    attr_hits = {k: 0 for k in TOP_KS}
    with torch.inference_mode():
        for start in range(0, total, config["val_batch_size"]):
            pos = indices[start:start + config["val_batch_size"]]
            batch = batch_from_index(pair_index, pos)
            src_idx = batch["source_indices"].long()
            tgt_idx = batch["target_indices"].long()
            source = image_embeddings[src_idx].float().to(device)
            attr_indices = batch["attr_indices"].to(device)
            signs = batch["signs"].to(device)
            conditions, mask = condition_embeddings(attr_indices, signs, device)
            query, _, _ = model(source, conditions, mask)
            scores = query @ gallery.T
            scores[torch.arange(len(src_idx), device=device), src_idx.to(device)] = -float("inf")
            ranking = torch.argsort(scores, descending=True, dim=1)[:, :max(TOP_KS)].cpu()
            for row_i, target_i in enumerate(tgt_idx.tolist()):
                for k in TOP_KS:
                    topk = ranking[row_i, :k].tolist()
                    exact_hits[k] += int(target_i in topk)
                    # Attribute success: at least one top-k image satisfies requested signed edits.
                    ok = False
                    for cand in topk:
                        cand_attrs = pair_index["attrs"][cand]
                        for a, s in zip(batch["attr_indices"][row_i], batch["signs"][row_i]):
                            if int(a) < 0:
                                continue
                            if int(cand_attrs[int(a)]) != int(s):
                                break
                        else:
                            ok = True
                            break
                    attr_hits[k] += int(ok)
    return {f"exact_R@{k}": exact_hits[k] / total for k in TOP_KS} | {f"attr_success@{k}": attr_hits[k] / total for k in TOP_KS}


def train_one_config(config, run_name):
    device = DEVICE
    model = GateSequentialComposer(
        edit_scale=config["edit_scale"], gate_max=config["gate_max"], residual_scale=config["residual_scale"],
        dropout=config["dropout"], gate_state=config["gate_state"],
    ).to(device)
    train_embeddings = F.normalize(image_caches["train"]["embeddings"].float(), dim=-1)
    valid_embeddings = F.normalize(image_caches["valid"]["embeddings"].float(), dim=-1)
    opt = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"])
    run_dir = RUN_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    metrics_rows = []
    best_score = -1
    best_path = run_dir / "best_model.pt"
    steps = config["epochs"] * config["steps_per_epoch"]
    step = 0
    for epoch in range(1, config["epochs"] + 1):
        model.train()
        for _ in range(config["steps_per_epoch"]):
            step += 1
            loss, parts = compute_train_loss(model, train_pairs, train_embeddings, config, device)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            if step % config["validate_every_steps"] == 0 or step == steps:
                val = validate_model(model, valid_pairs, valid_embeddings, config, device, config["val_max_queries"])
                row = {"epoch": epoch, "step": step, "loss": float(loss.detach()), **parts, **val}
                metrics_rows.append(row)
                print(run_name, row)
                score = val["attr_success@10"] + val["exact_R@10"]
                if score > best_score:
                    best_score = score
                    torch.save({"model_state": model.state_dict(), "config": config, "attributes": attributes}, best_path)
    pd.DataFrame(metrics_rows).to_csv(run_dir / "metrics.csv", index=False)
    return {"run_name": run_name, "run_dir": str(run_dir), "best_path": str(best_path), "best_score": best_score}


## Step 5 - Hyperparameter Search

The final search varies the same factors explored on the cluster: gate state, edit scale, gate maximum, residual scale, learning rate, and temperature. For short demonstrations we use fewer epochs; for full training we use the long profile.


In [ ]:
BASE_CONFIG = {
    "composer_type": "sequential_gate",
    "sampler_mode": "balanced_length",
    "batch_size": 128 if TRAIN_PROFILE == "short" else 256,
    "epochs": 3 if TRAIN_PROFILE == "short" else 60,
    "steps_per_epoch": 100 if TRAIN_PROFILE == "short" else 2000,
    "validate_every_steps": 100 if TRAIN_PROFILE == "short" else 4000,
    "val_max_queries": 512 if TRAIN_PROFILE == "short" else 4096,
    "val_batch_size": 128 if TRAIN_PROFILE == "short" else 256,
    "weight_decay": 1e-4,
    "lambda_target": 0.1,
    "lambda_source": 0.02,
    "false_negative_hamming": 2,
    "dropout": 0.1,
}

HP_CONFIGS = [
    {"config_id": "seq_l001", "gate_state": "current", "learning_rate": 3e-4, "temperature": 0.03, "residual_scale": 0.02, "edit_scale": 1.0, "gate_max": 1.5},
    {"config_id": "seq_l002", "gate_state": "source",  "learning_rate": 3e-4, "temperature": 0.03, "residual_scale": 0.02, "edit_scale": 1.0, "gate_max": 1.5},
    {"config_id": "seq_l003", "gate_state": "current", "learning_rate": 3e-4, "temperature": 0.03, "residual_scale": 0.02, "edit_scale": 0.75, "gate_max": 1.5},
    {"config_id": "seq_l004", "gate_state": "current", "learning_rate": 3e-4, "temperature": 0.03, "residual_scale": 0.02, "edit_scale": 1.25, "gate_max": 1.5},
    {"config_id": "seq_l005", "gate_state": "current", "learning_rate": 3e-4, "temperature": 0.03, "residual_scale": 0.02, "edit_scale": 1.0, "gate_max": 1.0},
    {"config_id": "seq_l006", "gate_state": "current", "learning_rate": 3e-4, "temperature": 0.03, "residual_scale": 0.02, "edit_scale": 1.0, "gate_max": 2.0},
    {"config_id": "seq_l007", "gate_state": "current", "learning_rate": 1e-4, "temperature": 0.03, "residual_scale": 0.02, "edit_scale": 1.0, "gate_max": 1.5},
    {"config_id": "seq_l008", "gate_state": "current", "learning_rate": 3e-4, "temperature": 0.05, "residual_scale": 0.02, "edit_scale": 1.0, "gate_max": 1.5},
    {"config_id": "seq_l009", "gate_state": "current", "learning_rate": 3e-4, "temperature": 0.03, "residual_scale": 0.0,  "edit_scale": 1.0, "gate_max": 1.5},
    {"config_id": "seq_l010", "gate_state": "current", "learning_rate": 3e-4, "temperature": 0.03, "residual_scale": 0.05, "edit_scale": 1.0, "gate_max": 1.5},
]
if HPSEARCH_LIMIT is not None:
    HP_CONFIGS = HP_CONFIGS[:HPSEARCH_LIMIT]

print("Training profile:", TRAIN_PROFILE)
print("Configs:", len(HP_CONFIGS))
for c in HP_CONFIGS:
    print(c)


In [ ]:
hp_rows = []
if RUN_TRAINING:
    for cfg_partial in HP_CONFIGS:
        cfg = {**BASE_CONFIG, **cfg_partial}
        run_name = f"gate_v3_{cfg['config_id']}_{TRAIN_PROFILE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        result = train_one_config(cfg, run_name)
        hp_rows.append({**cfg, **result})
    hp_df = pd.DataFrame(hp_rows).sort_values("best_score", ascending=False)
    hp_df.to_csv(RESULTS_DIR / "gate_v3_hpsearch_summary.csv", index=False)
else:
    summary_path = RESULTS_DIR / "gate_v3_hpsearch_summary.csv"
    hp_df = pd.read_csv(summary_path) if summary_path.exists() else pd.DataFrame()

hp_df.head(20)


## Step 6 - Official JSON Evaluation of the Learned Model

The final evaluation ranks the full test gallery for every source image provided by `celeba_evaluation.json`. We exclude the source image itself and compute Recall@K and Precision@K.


In [ ]:
def load_best_gate_model():
    if hp_df.empty:
        raise RuntimeError("No trained checkpoint found. Run training or provide a checkpoint summary.")
    row = hp_df.iloc[0]
    checkpoint = torch.load(row["best_path"], map_location=DEVICE, weights_only=False)
    config = checkpoint["config"]
    model = GateSequentialComposer(
        edit_scale=config["edit_scale"], gate_max=config["gate_max"], residual_scale=config["residual_scale"],
        dropout=config["dropout"], gate_state=config["gate_state"],
    ).to(DEVICE)
    model.load_state_dict(checkpoint["model_state"])
    model.eval()
    return model, config, row


def gate_compose_for_official(model, source, conditions):
    max_len = 3
    attr_idx = torch.full((1, max_len), -1, dtype=torch.long)
    sign_tensor = torch.zeros((1, max_len), dtype=torch.int8)
    for i, (sign, attr) in enumerate(conditions[:max_len]):
        attr_idx[0, i] = attribute_to_index[attr]
        sign_tensor[0, i] = sign
    cond, mask = condition_embeddings(attr_idx.to(DEVICE), sign_tensor.to(DEVICE), DEVICE)
    with torch.inference_mode():
        q, _, _ = model(source.to(DEVICE), cond, mask)
    return q.cpu()

learned_df = None
if RUN_OFFICIAL_EVAL and RUN_TRAINING and not hp_df.empty:
    best_model, best_config, best_row = load_best_gate_model()
    learned_df, learned_records = evaluate_method_on_json(
        "gate_v3_learned_sequential",
        lambda source, cond: gate_compose_for_official(best_model, source, cond),
    )
    learned_df.to_csv(RESULTS_DIR / "gate_v3_official_per_query.csv", index=False)
    print("Best config:", best_config)
    print("Official macro R@10:", learned_df["Recall@10"].mean())
else:
    path = RESULTS_DIR / "gate_v3_official_per_query.csv"
    learned_df = pd.read_csv(path) if path.exists() else None
    print("Skipped learned official evaluation or no learned result yet.")

learned_df


## Cosine Geometry: Why the Delta Correction Can Help

![Toy CLIP/cosine view](../final_best_system/explanations/toy_vector_correction_clip_cosine.png)

Stessa direzione. Per cosine similarity sono praticamente uguali.

Il punto chiave: CLIP retrieval non chiede "quanto sono vicino come coordinate assolute?", ma:

```text
qual è l'immagine con embedding che ha angolo/cosine più alto rispetto a q_final?
```

This is why adding the arithmetic delta can be useful even if it looks odd in Euclidean coordinates. After normalization, retrieval ranks gallery images by angle:

```text
score_i = cosine(q_final, image_embedding_i)
topK    = argsort(score_i, descending=True)[:K]
```


## Loading the Final Checkpoint From the Repo


In [ ]:
# This cell uses repo-relative paths, so it works after cloning/pulling the repo.
import os
import sys
from pathlib import Path

def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        if (candidate / "final_best_system").is_dir() and (candidate / "cluster").is_dir():
            return candidate
    raise RuntimeError("Could not find repo root containing final_best_system/ and cluster/.")

PROJECT_ROOT = find_project_root()
CLUSTER_ROOT = PROJECT_ROOT / "cluster"
FINAL_DIR = PROJECT_ROOT / "final_best_system"

os.environ["DL_PROJECT_ROOT"] = str(CLUSTER_ROOT)
sys.path.insert(0, str(CLUSTER_ROOT / "scripts"))
sys.path.insert(0, str(CLUSTER_ROOT / "orchestrator"))

from learned_gate_core import (
    condition_embeddings as learned_condition_embeddings,
    load_model_checkpoint,
    load_prompt_embedding_cache,
)
from project_core import choose_device, load_torch, parse_query, read_attribute_table

FINAL_CHECKPOINT_PATH = FINAL_DIR / "weights" / "best_val_official_like_at10.pt"
assert FINAL_CHECKPOINT_PATH.exists(), f"Missing checkpoint: {FINAL_CHECKPOINT_PATH}"

DEVICE = choose_device("auto")
final_model, final_checkpoint = load_model_checkpoint(FINAL_CHECKPOINT_PATH, DEVICE)
final_model.eval()
final_config = final_checkpoint["config"]

prompt_cache_path = CLUSTER_ROOT / final_config["prompt_cache_path"]
if not prompt_cache_path.exists():
    prompt_cache_path = FINAL_DIR / "embeddings" / Path(final_config["prompt_cache_path"]).name
assert prompt_cache_path.exists(), f"Missing prompt cache: {prompt_cache_path}"

final_prompt_cache = load_prompt_embedding_cache(prompt_cache_path)
text_bank = load_torch(FINAL_DIR / "embeddings" / "attribute_text_embeddings.pt")
attributes, _, _ = read_attribute_table()
attribute_to_index = {name: i for i, name in enumerate(attributes)}

print("Loaded final checkpoint:", FINAL_CHECKPOINT_PATH)
print("Checkpoint config_id:", final_config.get("config_id"))
print("Prompt cache:", prompt_cache_path)
print("Device:", DEVICE)


## Final System Function

The function below implements the winning system:

```text
q_final = normalize(q_model + beta * (q_sum - source))
```

where `q_model` comes from the trained sequential gate and `q_sum` is generic contrastive CLIP arithmetic.


In [ ]:
def _condition_tensors_for_query(conditions, batch_size, device):
    max_len = max(1, len(conditions))
    attrs = torch.full((batch_size, max_len), -1, dtype=torch.long, device=device)
    signs = torch.zeros((batch_size, max_len), dtype=torch.int8, device=device)
    for pos, (sign, attr) in enumerate(conditions):
        attrs[:, pos] = attribute_to_index[attr]
        signs[:, pos] = int(sign)
    return attrs, signs


def learned_gate_query(source_embeddings, conditions):
    source_embeddings = F.normalize(source_embeddings.float().to(DEVICE), dim=-1)
    attrs, signs = _condition_tensors_for_query(conditions, len(source_embeddings), DEVICE)
    cond, mask = learned_condition_embeddings(
        final_prompt_cache,
        attrs,
        signs,
        DEVICE,
        str(final_config.get("condition_mode", "signed_direction")),
    )
    with torch.inference_mode():
        q_model, alpha, _ = final_model(source_embeddings, cond, mask)
    return F.normalize(q_model, dim=-1), alpha


def generic_sum_query(source_embeddings, conditions):
    source_embeddings = F.normalize(source_embeddings.float().to(DEVICE), dim=-1)
    directions = F.normalize(text_bank["directions"].float().to(DEVICE), dim=-1)
    edit = torch.zeros_like(source_embeddings)
    for sign, attr in conditions:
        edit = edit + int(sign) * directions[attribute_to_index[attr]].unsqueeze(0)
    edit = F.normalize(edit, dim=-1)
    return F.normalize(source_embeddings + edit, dim=-1)


def final_model_plus_delta_query(source_embeddings, query_text, beta=1.0):
    conditions = parse_query(query_text)
    source_embeddings = F.normalize(source_embeddings.float().to(DEVICE), dim=-1)
    q_model, alpha = learned_gate_query(source_embeddings, conditions)
    q_sum = generic_sum_query(source_embeddings, conditions)
    q_final = F.normalize(q_model + beta * (q_sum - source_embeddings), dim=-1)
    return q_final, {"conditions": conditions, "alpha": alpha.detach().cpu(), "q_model": q_model.detach().cpu(), "q_sum": q_sum.detach().cpu()}


# Minimal smoke test with a random normalized vector. The real evaluation cells
# use CLIP image embeddings from the dataset.
_dummy_source = F.normalize(torch.randn(1, 512), dim=-1)
_dummy_q, _dummy_info = final_model_plus_delta_query(_dummy_source, "+Smiling, +Eyeglasses")
print("Final query shape:", tuple(_dummy_q.shape))
print("Parsed conditions:", _dummy_info["conditions"])
print("Learned gate weights:", _dummy_info["alpha"].numpy().round(3).tolist())


## Final Official Results Included in the Repo

The official JSON evaluation is already saved in `final_best_system/results/clean_report/`. The table below compares:

1. assignment vanilla baseline: `direct_sum`;
2. strongest zero-shot CLIP baseline: `contrastive_sequential`;
3. final proposed system: `model_plus_generic_delta_100`.

The assignment metrics are `Recall@1/5/10` and `Precision@1/5/10`.


In [ ]:
from IPython.display import Image, display

macro_metrics = pd.read_csv(FINAL_DIR / "results" / "clean_report" / "overall_metrics_macro.csv")
micro_metrics = pd.read_csv(FINAL_DIR / "results" / "clean_report" / "overall_metrics_micro.csv")
per_query_r10 = pd.read_csv(FINAL_DIR / "results" / "clean_report" / "per_query_recall10_three_systems.csv")

print("Macro metrics: each query counts equally.")
display(macro_metrics)

print("Micro metrics: every source-query case counts equally.")
display(micro_metrics)

display(Image(filename=str(FINAL_DIR / "results" / "clean_report" / "overall_metrics_three_systems.png")))
display(Image(filename=str(FINAL_DIR / "results" / "clean_report" / "per_query_recall10_three_systems.png")))

per_query_r10


## Results Tables and Plots

We compare zero-shot baselines and the learned model using the official JSON metrics. Recall@K is primary, Precision@K secondary.


In [ ]:
all_result_frames = []
for method, df in baseline_results.items():
    all_result_frames.append(df.copy())
if learned_df is not None:
    all_result_frames.append(learned_df.copy())

if all_result_frames:
    all_per_query = pd.concat(all_result_frames, ignore_index=True)
    summary_rows = []
    for method, group in all_per_query.groupby("method"):
        row = {"method": method}
        for k in TOP_KS:
            row[f"macro_Recall@{k}"] = group[f"Recall@{k}"].mean()
            row[f"macro_Precision@{k}"] = group[f"Precision@{k}"].mean()
        summary_rows.append(row)
    summary = pd.DataFrame(summary_rows).sort_values("macro_Recall@10", ascending=False)
    summary.to_csv(RESULTS_DIR / "official_summary.csv", index=False)
    display(summary)

    ax = summary.set_index("method")[["macro_Recall@1", "macro_Recall@5", "macro_Recall@10"]].plot(kind="bar", figsize=(10, 5))
    ax.set_title("Official JSON Macro Recall")
    ax.set_ylabel("Recall")
    ax.tick_params(axis="x", rotation=25)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "official_macro_recall.png", dpi=160)
    plt.show()
else:
    print("No results available yet.")


In [ ]:
if all_result_frames:
    pivot = all_per_query.pivot_table(index=["query_id", "query"], columns="method", values="Recall@10")
    display(pivot)
    pivot.plot(kind="bar", figsize=(14, 6))
    plt.title("Recall@10 by Query")
    plt.ylabel("Recall@10")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "per_query_recall10.png", dpi=160)
    plt.show()


## Discussion: Findings and Pivots

The main finding is that CLIP arithmetic is not just a toy baseline. The assignment-style direct sum is a useful lower bound, but contrastive directions and sequential normalization make zero-shot CLIP much stronger.

Development pivots:

1. **Direct arithmetic baseline.** Required by the assignment and used as the vanilla reference.
2. **Contrastive sequential baseline.** Improved by using `t_positive - t_negative` and normalizing after each edit.
3. **`gate_v1` residual-only model.** Learned something, but had to rediscover CLIP edit directions from scratch.
4. **`gate_v2` additive gate.** Used contrastive directions with learned weights.
5. **`gate_v3` sequential gate.** Learned a source-conditioned version of the strongest sequential arithmetic geometry.
6. **Final hybrid system.** The best report system combines the learned gate with a generic CLIP arithmetic delta:

```text
q_final = normalize(q_model + 1.0 * (q_sum - source))
```

Current official JSON results:

```text
Assignment direct_sum Macro Recall@10:          0.1084
Strong contrastive_sequential Macro Recall@10:  0.1871
Final model+delta Macro Recall@10:              0.2827

Assignment direct_sum Micro Recall@10:          0.1248
Strong contrastive_sequential Micro Recall@10:  0.1665
Final model+delta Micro Recall@10:              0.2386
```

The final system improves strongly on local visual edits such as eyeglasses, smile, makeup, and mustache. The hardest remaining attributes are global/correlated ones such as `Male`, `Young`, and `Chubby`. These are difficult because CLIP directions encode broad demographic/semantic shifts, while the official target sets often require subtle attribute changes without destroying identity and other non-query attributes.

## Reproducibility and Academic Integrity

This notebook uses standard libraries (`torch`, `torchvision`, `transformers`, `pandas`, `matplotlib`) and custom code written for this assignment. It does not depend on an external third-party project repository.

CLIP ViT-B/32 is the required Hugging Face model. CLIP remains frozen; only the small gate/residual composition module is trained.

The official JSON is used only for final evaluation, not for training. Training pairs are generated from the CelebA train split using same-identity pairs and attribute differences.
